# EDA for Cost-Sensitive ML

In [ ]:
import os
from pathlib import Path

import boto3
import botocore.exceptions
import joblib
import numpy as np
import pandas as pd
import pyarrow as pa
import sklearn.utils as skut
from dotenv import load_dotenv
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [ ]:
PROJ_ROOT = Path.cwd().parent

In [ ]:
assert load_dotenv(dotenv_path=PROJ_ROOT.parent / ".env")

In [ ]:
import cc_churn.visualization as vzu
import r2.io_utils as r2io
from cc_churn.transformers import CategoryCombiner2
from utils.df_utils import highlight_abs_greater, show_df

## About

Machine Learning (ML) model development using tree-based models.

In order to account for the strong class imbalance, nested cross-validation (CV) is performed using `TunedThresholdClassifierCV()` in `cross_validate()` on the combined training and validation data.

This gives the best classifier decision threshold (via inner CV).

Outer CV is used to compare the choices of

1. classifier
2. feature selection
3. feature pre-processing (based on the selected features)

This approach follows the example in the [`scikit-learn` documentation](https://scikit-learn.org/stable/auto_examples/model_selection/plot_tuned_decision_threshold.html#tuning-the-decision-threshold).

These three choices are compared using ML experiments. Details are explained later.

The out-of-sample performance of the best combination of the three choices above is evaluated against the test data, which was not used in model development.

### Outputs

Based on the project deliverables in [this project's scoping document](https://github.com/edesz/credit-card-churn/blob/main/references/01_proposal.md#project-deliverables), this notebook produces following outputs

1. The best trained ML model is used to make predictions on the
   - all available data and these predictions and probabilities are exported to a file. These predictions will be used in the next step to calculate business metrics on all existing customers who have churned.
2. The best ML model object is exported after training it on
   - all available data

## User Inputs

In [ ]:
# R2 data bucket details
# # name of train data key (file) in private R2 bucket
r2_key_train = "train_data.parquet.gzip"
# # name of validation data key (file) in private R2 bucket
r2_key_val = "validation_data.parquet.gzip"
# # name of test data key (file) in private R2 bucket
r2_key_test = "test_data.parquet.gzip"

# datatypes for categorical and ordinal columns
dtypes_ordinals = {
    "gender": "string[pyarrow]",
    "income_category": "string[pyarrow]",
    "education_level": "string[pyarrow]",
}
dtypes_categoricals = {
    "marital_status": "string[pyarrow]",
    "card_category": "string[pyarrow]",
}

# EDA
threshold_correlation = 0.55

label = "is_churned"

In [ ]:
account_id = os.getenv("ACCOUNT_ID")
access_key_id = os.getenv("ACCESS_KEY_ID_USER2")
secret_access_key = os.getenv("SECRET_ACCESS_KEY_USER2")
bucket_name = os.getenv("BUCKET_NAME")

s3_client = boto3.client(
    "s3",
    endpoint_url=f"https://{account_id}.r2.cloudflarestorage.com",
    aws_access_key_id=access_key_id,
    aws_secret_access_key=secret_access_key,
    region_name="auto",
)

## Load Data

### Data for Model Validation

Load the training data

In [ ]:
%%time
df_train = (
    r2io.pandas_read_parquet_r2(s3_client, bucket_name, r2_key_train)
    .astype(dtypes_ordinals)
    .astype(dtypes_categoricals)
)
print(f"Loaded {len(df_train):,} rows of training data")
dfd = show_df(df_train)
with pd.option_context('display.max_columns', None):
    display(df_train.head(1))

Load the validation data

In [ ]:
%%time
df_val = (
    r2io.pandas_read_parquet_r2(s3_client, bucket_name, r2_key_val)
    .astype(dtypes_ordinals)
    .astype(dtypes_categoricals)
)
print(f"Loaded {len(df_val):,} rows of validation data")
_ = show_df(df_val)
with pd.option_context('display.max_columns', None):
    display(df_val.head(1))

Get the combined training+validation data split

In [ ]:
%%time
df_train_val = pd.concat([df_train, df_val], ignore_index=True)
print(f"Obtained {len(df_train_val):,} rows of training+validation data")

## Separate Features from Target

In [ ]:
X_train_val = df_train_val.drop(columns=[label])
y_train_val = df_train_val[label]

## EDA

### Class Imbalance

Show the class imbalance

In [ ]:
display(y_train_val.value_counts(normalize=True).to_frame())

Use the class imbalance to get the class_weights for cost-sensitive learning

In [ ]:
# utilize inverse of class distribution (class imbalance ratio)
class_weights_train_val = skut.class_weight.compute_class_weight(
    class_weight="balanced",
    classes=y_train_val.squeeze().unique().to_numpy(),
    y=y_train_val.squeeze().to_numpy(),
)
class_weights_train_val = {
    c: float(w) for w, c in zip(class_weights_train_val, [0, 1])
}

class_weights_train_val

Get `scale_pos_weight` for use in `XGBClassifier`

In [ ]:
scale_pos_weight_train_val = (
    class_weights_train_val[1] / class_weights_train_val[0]
)
print(scale_pos_weight_train_val)

### Correlated Features

Get the correlation between all numerical features and then show combinations of features that are correlated above a threshold of 0.55

In [ ]:
%%time
# correlation between all numerical features
df_corr = X_train_val[
    list(X_train_val.select_dtypes(['int64', 'double']))+['customer_age']
].corr()
lower_df = df_corr.where(~np.triu(np.ones_like(df_corr, dtype=bool), k=0))
display(
    lower_df.style.apply(highlight_abs_greater, threshold=0.55, axis=None)
)

# Filter for combinations of features that are highly correlated
high_corr_pairs = []
for i in range(len(df_corr.columns)):
    for j in range(i + 1, len(df_corr.columns)):
        col1 = df_corr.columns[i]
        col2 = df_corr.columns[j]
        corr_value = df_corr.loc[col1, col2]
        if abs(corr_value) > threshold_correlation:
            high_corr_pairs.append(
                {
                    'feature_1': col1,
                    'feature_2': col2,
                    'correlation': corr_value,
                }
            )
df_correlated_features = (
    pd.DataFrame.from_records(high_corr_pairs)
    .sort_values(by=['correlation'], ascending=False, ignore_index=True)
)
display(
    df_correlated_features
    .style
    .set_properties(subset=['correlation'], **{'background-color': 'yellow'})
)

Feature multicollinearity distorts feature importance, reduces the stability of a model, and hides the true predictive power of individual features. Tree-based models are robust to multicollinearity for prediction accuracy. However, this correlation causes issues when trying to interpret their predcitions. So, we will keep one feature from every pair of highly-correlated features. There are two pairs of correlated features as shown in the above table. We will keep one of these (eg. `feature_1` or `feature_2`) and exclude the other.

### Non-Numerical Features with Rare Categories

Show the number of unique values in all categorical and ordinal features

In [ ]:
(
    X_train_val[list(X_train_val.select_dtypes(["category", "string"]))]
    .nunique()
    .reset_index()
    .rename(columns={"index": "feature_name", 0: "num_unique_values"})
)

High cardinality is not observed in any of the categorical or ordinal features. Categorical features can be one-hot encoded with minimal feature expansions. This has several benefits

1. it keeps the encoded data dense, thereby improving the speed of model training
2. it is easier to interpret
3. it avoids the [curse of dimensionality when performing one-hot encoding](https://apxml.com/courses/intro-feature-engineering/chapter-3-encoding-categorical-features/high-cardinality-features)

So, all categorical and ordinal features can be used in model development.

Show the frequency of sub-categories per ordinal and categorical fearture

In [ ]:
%%time
for c in [
    'gender',
    "income_category",
    "education_level",
    "card_category",
    "marital_status",
    'dependent_count'
]:
    display(
        X_train_val[c]
        .value_counts(normalize=True)
        .mul(100)
        .to_frame()
        .merge(
            X_train_val[c]
            .value_counts()
            .to_frame(),
            left_index=True,
            right_index=True,
            how='left',
        )
        .assign(feature=c)
    )

As we can see, there are some categorical and ordinal features with rare categories (categories that do not occur frequently in the dataset). They result in unnecessary high-dimensional, sparse transformed features that lead to models overfitting to noise instead of learning meaningful patterns. So, we will need to either drop or group these rare categories to improve the usefulness of the feature.

#### Re-Grouping

Based on the above category frequencies, we will transform these two types of features as follows

1. `marital_status`
   - combine `Unknown` and `Divorced` into `Other`
   - this causes all sub-categories to occur with >15% frequency in the combined train+validation data, and eliminates two which each have a frequency of <~7.4%
2. `education_level`
   - combine the two lowest frequency sub-categories `Post-Graduate` and `Doctorate` into a category called `Post-Graduate`, as both levels of education are similar to each other and are above all other sub-categories
   - this causes all sub-categories to occur with >10% frequency in the combined train+validation data, and eliminates two which each have a frequency of <~5%
3. `income_category`
   - combine `80K-120K` and `$120K +` since these are generally high-income customers
   - similar to `education_level`, this causes all sub-categories to occur with >10% frequency in the combined train+validation data, and eliminates one which has a frequency of <~7%
4. `dependent_count`
   - although this feature appears as a count, we will treat this as a categorical. The effect of dependents on churn is likely not linear. For example, the difference in lifestyle/financial need between `0` and `1` dependent might be similar to the difference between `4` and `5`, or entirely different. Categorical encoding allows a model (like Random Forest or XGBoost) to learn specific, non-linear impacts for each specific count (e.g. `0`, `1`, `2`, `3`, `4`, `5`).
   - - combine `4` and `5` into a new category `4+`, which causes all sub-categories to occur with >=9% frequency in the combined train+validation data, and eliminates one which has a frequency of ~4.2%
5. `card_category`
   - the value `Blue` accounts for approximately 93% of all customers in the combined train+validation data. This is almost entirely a single-valued feature that does not have much predictive power. So, we will drop this feature.

Below is a `scikit-learn` `Pipeline` that uses a custom transformer `CategoryCombiner2` to combine the categories for the four features from above

In [ ]:
cat_ord_grouper = CategoryCombiner2().set_output(transform="pandas")
transformers_preprocessors = [("catgroup", cat_ord_grouper)]
pipe_cat_ord_transformer = Pipeline(transformers_preprocessors).set_output(
    transform="pandas"
)

This custom category combining transformer is applied below to the combined train+validation data and the test data

In [ ]:
# train custom transformer using combined train+validation data
_ = pipe_cat_ord_transformer.fit(X_train_val)

# use trained transfromer to get transformed column names
columns_transformed = pipe_cat_ord_transformer.get_feature_names_out(
    input_features=list(X_train_val)
).tolist()

# apply trained transformer to perform category grouping on combined
# train+validation data
X_train_val = pd.DataFrame(
    pipe_cat_ord_transformer.transform(X_train_val),
    columns=columns_transformed,
    index=X_train_val.index,
)

Below are the transformed frequencies of sub-categories per ordinal and categorical fearture, excluding the `card_category` feature which will be dropped

In [ ]:
for c in [
    "income_category",
    "education_level",
    "marital_status",
    "dependent_count",
]:
    display(
        X_train_val[c]
        .value_counts(normalize=True)
        .mul(100)
        .to_frame()
        .merge(
            X_train_val[c].value_counts().to_frame(),
            left_index=True,
            right_index=True,
            how="left",
        )
        .assign(feature=c)
    )

As expected, all sub-categories in the categorical features now occur with a frequency of at least 10% for three of the grouped features and 9% for the fourth feature.

### Handling Sensitive Features

Below are the sensitive features in the data

1. `age`
2. `gender`
3. `marital_status`
4. `dependent_count`
4. `education_level`
5. `income_category`

According to [Kamiran and Calders](https://link.springer.com/article/10.1007/s10115-011-0463-8), sensitive featyues can be used as predictive features. However, they must be handled through specific preprocessing techniques to avoid illegal and unethical discrimination of customers during a targeted marketing campaign (as is the overall use-case for this project) based on theses sensitive.

One recommended approach to ensure the predictive model does not unfairly discriminate based on age is to remove the sensitive feature and other features that correlate most with it, reducing the model's reliance on potentially discriminatory information. This approach will be used for `customer_age`, so it will be excluded from model development. Since `age` is not strongly correlated to other features as shown above, no other features need to be dropped in order to follow this approach.

`gender` is a *balanced* categorical feature that has two sub-categories that occur with almost the same frequency. So, we don't need to group any sub-categories and we will keep this feature unchanged.

We will run ML experiments to determine if the other sensitive features (which are either categorical or ordinal) can be dropped without negatively impacting model performance.

### Feature Lists by Type

Based on the EDA from above, features using in ML development should exclude the following

1. correlated numerical features
2. single-valued feature (`card_category`)
3. identifier features (`clientnum`)
4. sensitive feature (`customer_age`)

With this in mind, the ordinal, categorical and numerical features to be used in validation and evaluation are shown below

In [ ]:
ordinal_features = [
    "income_category",
    "education_level",
]

categorical_features = [
    "gender",
    "marital_status",
    "dependent_count",
]

# keep
# - 'credit_limit' and exclude "avg_open_to_buy" which is correlated
# - 'total_revolv_bal' and exclude "avg_utilization_ratio" which is correlated
numeric_features_1 = [
    "months_on_book",
    "num_products",
    "months_inactive_12_mon",
    "contacts_count_12_mon",
    "total_amt_chng_q4_q1",
    "total_trans_amt",
    "total_trans_ct",
    "total_ct_chng_q4_q1",
    "credit_limit",
    "total_revolv_bal",
]

# keep
# - 'avg_open_to_buy' and exclude "credit_limit" which is correlated
# - 'avg_utilization_ratio' and exclude "total_revolv_bal" which is correlated
numeric_features_2 = [
    "months_on_book",
    "num_products",
    "months_inactive_12_mon",
    "contacts_count_12_mon",
    "total_amt_chng_q4_q1",
    "total_trans_amt",
    "total_trans_ct",
    "total_ct_chng_q4_q1",
    "avg_open_to_buy",
    "avg_utilization_ratio",
]

Below is a summary of the features after transformation

In [ ]:
_ = show_df(
    X_train_val[ordinal_features + categorical_features + numeric_features_1]
)

## Conclusions

There are two combinations of numerical features that are correlated to each other (correlation coefficient greater than 0.55). For model interpretability, one feature from each combination should be used.

If rare categories in categorical and ordinal features are grouped then the minimum category frequency can be increased to at least 9% for `dependent_count` and 10% for the other three features. This will help improve the predictive power of these features. The `card_category` feature should be dropped since it is mostly single-valued.

Finally, based on the class imbalance, the `scale_pos_weight` used by `XGBClassifier` should be 5.223 during model training.